In [30]:
import pandas as pd
df_network = pd.read_csv("problem1.csv")

Cost = {}
for i in df_network.index:
    Cost[(df_network['from'][i],df_network['to'][i])] = df_network['cost ($/bbl)'][i]

In [32]:
Arcs = {(i,j) for (i,j) in Cost}

In [62]:
Supply = { 1 : -6000, 2 : -4500, 3 : 0, 4 : 3500, 5 : 4000, 6: 3000 }

In [63]:
Nodes = range(1,7)

In [65]:
from docplex.mp.model import Model
mdl = Model()

In [66]:
# variables
leg = mdl.continuous_var_dict(Arcs, lb=0, name='leg')

In [67]:
# objective
mdl.minimize(mdl.sum(Cost[i,j]*leg[i,j] for (i,j) in Arcs))

In [68]:
# flow conservation constraint
for j in Nodes:
    inflow = mdl.sum(leg[i,j] for i in Nodes if (i,j) in Arcs)
    outflow = mdl.sum(leg[j,i] for i in Nodes if (j,i) in Arcs)
    mdl.add_constraint(inflow - outflow == Supply[j])

In [71]:
# max throughput for site 3
mdl.add_constraint(mdl.sum(leg[i,3] for i in Nodes if (i,3) in Arcs) <= 3000)

docplex.mp.LinearConstraint[](leg_2_3+leg_1_3,LE,3000)

In [72]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00335598,status='optimal')

In [73]:
mdl.print_solution()

objective: 41750.000
status: OPTIMAL_SOLUTION(2)
  leg_2_4=3500.000
  leg_1_6=3000.000
  leg_2_5=1000.000
  leg_1_3=3000.000
  leg_3_5=3000.000
